In [0]:
spark.sql("""
          create schema if not exists 
          dbw_netflix_de_2026.netflix_gold
          """)

silver_table = "dbw_netflix_de_2026.netflix_silver.netflix_titles"

df_silver = spark.table(silver_table)
print("Silver records:", df_silver.count())


Silver records: 6232


In [0]:
from pyspark.sql.functions import col,count

df_content_summary = (
    df_silver
    .groupby("type")
    .agg(count("*").alias("total_titles"))
    .orderBy(col("total_titles").desc())
  
)

gold_table_1 = ("dbw_netflix_de_2026.netflix_gold.content_summary")
(
    df_content_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table_1)
)

print("gold table created")
display(spark.table(gold_table_1))

gold table created


type,total_titles
Movie,4263
TV Show,1969


In [0]:
df_year_summary = (
    df_silver
    .filter(col("release_year"). isNotNull())
    .groupby("release_year","type")
    .agg(count("*").alias("total.titles"))
    .orderBy("release_year", "type")
)


gold_table_2 =(
    "dbw_netflix_de_2026.netflix_gold.release_year_summary"

    )

(
    df_year_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table_2)

)

print("gold table created")
display(spark.table(gold_table_2))

gold table created


release_year,type,total.titles
1925,TV Show,1
1942,Movie,2
1943,Movie,3
1944,Movie,2
1945,Movie,3
1946,Movie,2
1946,TV Show,1
1947,Movie,1
1954,Movie,1
1955,Movie,1


In [0]:

from pyspark.sql.functions import trim, when

df_rating_summary = (
    df_silver
    .withColumn(
        "rating_clean",
        when(
            col("rating").isNull() |
            (trim(col("rating")) == ""),
            "Unknown"
        ).otherwise(trim(col("rating")))
    )
    .groupBy("rating_clean")
    .agg(count("*").alias("total_titles"))
    .orderBy(col("total_titles").desc())
)

gold_table_3 = (
    "dbw_netflix_de_2026.netflix_gold.rating_summary"
)

(
    df_rating_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table_3)
)

print("Gold Table 3 created!")
display(spark.table(gold_table_3))


Gold Table 3 created!


rating_clean,total_titles
TV-MA,2026
TV-14,1698
TV-PG,700
R,508
PG-13,286
NR,218
PG,184
TV-Y7,169
TV-G,149
TV-Y,143


In [0]:

from pyspark.sql.functions import avg, round as spark_round

df_movie_duration = (
    df_silver
    .filter(
        (col("type") == "Movie") &
        (col("duration_minutes").isNotNull())
    )
    .groupBy("release_year")
    .agg(
        count("*").alias("movie_count"),
        spark_round(
            avg("duration_minutes"), 2
        ).alias("avg_duration_minutes")
    )
    .orderBy("release_year")
)

gold_table_4 = (
    "dbw_netflix_de_2026.netflix_gold.movie_duration_summary"
)

(
    df_movie_duration.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table_4)
)

print("Gold Table 4 created!")
display(spark.table(gold_table_4).limit(20))


Gold Table 4 created!


release_year,movie_count,avg_duration_minutes
1942,2,35.0
1943,3,62.67
1944,2,58.0
1945,3,51.33
1946,2,76.0
1947,1,42.0
1954,1,120.0
1955,1,111.0
1956,1,98.0
1958,2,111.5


In [0]:

gold_tables = [
    "content_summary",
    "release_year_summary",
    "rating_summary",
    "movie_duration_summary"
]

for table_name in gold_tables:
    full_name = (
        f"dbw_netflix_de_2026.netflix_gold.{table_name}"
    )

    print(

        table_name,
        "->",
        spark.table(full_name).count(),
        "rows"
    )


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6782709926871644>, line 17
      9 for table_name in gold_tables:
     10     full_name = (
     11         f"dbw_netflix_de_2026.netflix_gold.{table_name}"
     12     )
     14     print(
     15         table_name,
     16         "->",
---> 17         spark.table(full_name).count(),
     18         "rows"
     19     )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:381, in DataFrame.count(self)
    380 def count(self) -> int:
--> 381     table, _ = self.agg(F._invoke_function("count", F.lit(1)))._to_table()  # type: ignore[operator]
    382     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2168, in DataFrame._to_table(self)
   2166 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   2167     query 

In [0]:

spark.sql("""
SHOW TABLES IN dbw_netflix_de_2026.netflix_gold
""").show(truncate=False)


+------------+----------------------+-----------+
|database    |tableName             |isTemporary|
+------------+----------------------+-----------+
|netflix_gold|content_summary       |false      |
|netflix_gold|movie_duration_summary|false      |
|netflix_gold|rating_summary        |false      |
|netflix_gold|release_year_summary  |false      |
+------------+----------------------+-----------+



In [0]:

gold_tables = [
    "content_summary",
    "release_year_summary",
    "rating_summary",
    "movie_duration_summary"
]

for table_name in gold_tables:
    full_name = (
        f"dbw_netflix_de_2026.netflix_gold.{table_name}"
    )

    print(
        table_name,
        "->",
        spark.table(full_name).count(),
        "rows"
    )


content_summary -> 2 rows
release_year_summary -> 115 rows
rating_summary -> 15 rows
movie_duration_summary -> 70 rows


In [0]:

gold_content = spark.table(
    "dbw_netflix_de_2026.netflix_gold.content_summary"
)

display(gold_content)

print(
    "Silver records:",
    spark.table(
        "dbw_netflix_de_2026.netflix_silver.netflix_titles"
    ).count()
)

print(
    "Gold content total:",
    gold_content.agg(
        {"total_titles": "sum"}
    ).collect()[0][0]
)


type,total_titles
Movie,4263
TV Show,1969


Silver records: 6232
Gold content total: 6232


In [0]:

silver_table = "dbw_netflix_de_2026.netflix_silver.netflix_titles"

df_silver = spark.table(silver_table)

print("Silver records:", df_silver.count())


Silver records: 6232
